In [1]:
#import seaborn as sns
import pandas as pd
from pylab import *
%matplotlib inline
#from scipy.stats import gaussian_kde
from matplotlib.colors import ListedColormap
#from scipy.spatial import ConvexHull
#import statistics
#from scipy.spatial.distance import mahalanobis
from preprocessing_aux import *

In [2]:
infile = 'dataset_file.csv'
outfile = 'dataset_file_transformed.csv'
df = pd.read_csv(infile)

In [3]:
df[df['veg_class'] == 1]['B'].mean()

0.7116024883849122

In [4]:
df['rc'] = df['R'] / (df['R'] + df['G'] + df['B'])
df['gc'] = df['G'] / (df['R'] + df['G'] + df['B'])

In [5]:
# --------------------------------------------------------------------------------------
# extending pada data frame by rc/gc, rc+gc, luminance Y, perceived lightness L, z_score_Y, z_score_L 
# Excess indecies ExG, ExR, ExB, indices ExGmExR, Ikaw, MGRVI, and GLI 
# --------------------------------------------------------------------------------------
# blue chromaticity
df['bc'] = 1-(df['rc']+df['gc'])

# ration rc/gc
df['rc/gc'] = df['rc']/df['gc']

# sum rc+gc
df['rc+gc'] = df['rc']+df['gc']

# excess indecies
ExG = [excess_index(G=val, R=np.array(df['R'])[i], B=np.array(df['B'])[i], convert_to_255=False)
       for i, val in enumerate(np.array(df['G']))]
df['ExG'] = ExG

ExR = [excess_index(G=val, R=np.array(df['R'])[i], B=np.array(df['B'])[i], convert_to_255=False, excessband='red')
       for i, val in enumerate(np.array(df['G']))]
df['ExR'] = ExR

ExB = [excess_index(G=val, R=np.array(df['R'])[i], B=np.array(df['B'])[i], convert_to_255=False, excessband='blue')
       for i, val in enumerate(np.array(df['G']))]
df['ExB'] = ExB

df.head()

# index ExGmExR
df['ExGmExR'] = df['ExG']-df['ExR']

# index Ikaw - Kawashima Index
df['Ikaw'] = (df['R']-df['B'])/(df['R']+df['B'])

# MGRVI - modified green red vegetation index
df['MGRVI'] = ((df['G'])**2 - (df['R'])**2)/((df['G'])**2 + (df['R'])**2)

# GLI - green leaf index
df['GLI'] = (2*df['G'] - df['R'] - df['B'])/(2*df['G'] + df['R'] + df['B'])

# luminance Y
Y = [0.2126*sRGBtoLin(val) + 
     0.7152*sRGBtoLin(np.array(df['G'])[i]) + 
     0.0722*sRGBtoLin(np.array(df['B'])[i]) for i, val in enumerate(np.array(df['R']))]
df['Y'] = Y

# perceived lightness L
L = [YtoLstar(val) for val in df['Y']]
df['L'] = L

# z_score of the luminance Y
df['z_score_Y'] = 0
sites = set(df['site'])
for site in sites:
    condition = df['site'] == site
    subdata_site = df[df['site'].isin([site])]['Y']
    z_site = z_score(x_all=np.array(subdata_site))
    df.loc[condition, 'z_score_Y'] = z_site
    
# z_score of the perceived lightness L
df['z_score_L'] = 0
sites = set(df['site'])
for site in sites:
    condition = df['site'] == site
    subdata_site = df[df['site'].isin([site])]['L']
    z_site = z_score(x_all=np.array(subdata_site))
    df.loc[condition, 'z_score_L'] = z_site

print(df.head())

     site  y_pos  x_pos  veg_class         R         G         B  chm   
0  CG1-8A    513   7948         13  0.878431  0.870588  0.858824    1  \
1  CG1-8A    513   7949         13  0.913725  0.890196  0.870588    1   
2  CG1-8A    514   7948         13  0.874510  0.862745  0.854902    1   
3  CG1-8A    514   7949         13  0.894118  0.866667  0.843137    1   
4  CG1-8A    584   9232         12  0.364706  0.419608  0.282353    1   

   class_certainty        rc  ...       ExR       ExB   ExGmExR      Ikaw   
0                2  0.336842  ...  0.359216  0.331765 -0.355294  0.011287  \
1                2  0.341642  ...  0.389020  0.328627 -0.392941  0.024176   
2                2  0.337368  ...  0.361569  0.334118 -0.365490  0.011338   
3                2  0.343373  ...  0.385098  0.313725 -0.389020  0.029345   
4                2  0.341912  ...  0.090980 -0.024314  0.101176  0.127273   

      MGRVI       GLI         Y          L  z_score_Y  z_score_L  
0 -0.008968  0.001127  0.732043

In [6]:
df.to_csv(outfile, index=False)